# Fase 09 — Migracao: api-llm-embedded -> monorepo-ai-llm

**Data:** 2026-05-02  
**Contexto:** Migracao completa do projeto `api-llm-embedded` (npm workspaces, NestJS monolito multi-dominio, 5 MCP servers) para o monorepo `monorepo-ai-llm` (NestJS CLI monorepo, 5 apps separados).  
**Documento de referencia:** `spec/fases/fase-09-plano-migracao-api-llm-embedded.md`
**Padrao de raiz:** este projeto deve ser executado dentro de `/mnt/repositorio/workdir/repostorios/<repositorio>`.

---

## Indice

1. [Contexto e Diagnostico](#contexto)
2. [Fase 1 — Fundacao: npm Workspaces + Shared Packages](#fase1)
3. [Fase 2 — Infraestrutura de Banco de Dados](#fase2)
4. [Fase 3 — Migracao: users-api](#fase3)
5. [Fase 4 — Migracao: llm-ops-api](#fase4)
6. [Fase 5 — Migracao: sharepoint-api](#fase5)
7. [Fase 6 — Migracao: sync-api](#fase6)
8. [Fase 7 — Migracao: MCP Servers](#fase7)
9. [Fase 8 — Scripts, Config e Docs](#fase8)
10. [Fase 9 — CLAUDE.md](#fase9)
11. [Fase 10 — Validacao Final](#fase10)

## 1. Contexto e Diagnóstico <a id='contexto'></a>

### Projetos

| | Origem | Destino |
|---|---|---|
| **Caminho** | `/mnt/repositorio/workdir/repostorios/api-llm-embedded` | `/mnt/repositorio/workdir/repostorios/monorepo/monorepo-ai-llm` |
| **Build** | npm workspaces + `tsgo` | NestJS CLI + webpack |
| **Apps** | `apps/api` monolito + `apps/svcia/` | 5 NestJS apps separados |
| **DB** | TypeORM + migrations | Sem banco ainda |
| **Shared** | `packages/shared` | Não existe |
| **MCP** | 5 servers em `apps/svcia/` | Stub em `tools/mcp/` |

In [5]:
from pathlib import Path
import subprocess

# Diagnóstico inicial — executar antes de começar

ORIGEM = "/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO = "/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

print("=== Estrutura da ORIGEM ===")
for p in sorted(Path(ORIGEM).iterdir()):
    print(p.name)

print("\n=== Estrutura do DESTINO ===")
for p in sorted(Path(DESTINO).iterdir()):
    print(p.name)

print("\n=== Apps no DESTINO ===")
apps_dir = Path(DESTINO) / "apps"
if apps_dir.exists():
    for p in sorted(apps_dir.iterdir()):
        print(p.name)
else:
    print("Diretório apps não encontrado.")

print("\n=== Node.js e npm ===")
node_version = subprocess.run(["node", "--version"], capture_output=True, text=True)
npm_version = subprocess.run(["npm", "--version"], capture_output=True, text=True)

print(node_version.stdout.strip() if node_version.returncode == 0 else "node não disponível")
print(npm_version.stdout.strip() if npm_version.returncode == 0 else "npm não disponível")

=== Estrutura da ORIGEM ===
.agents
.claude
.copilot-tracking
.dockerignore
.env
.env.containers
.env.containers.example
.env.example
.git
.gitignore
.tarefas-agents
.vscode
AGENTS.md
CLAUDE.md
Dockerfile.langflow
Dockerfile.mcp
README.md
START_HERE.md
apps
audit-report.json
audit-results.json
azure-pipelines.yml
cagent.example.yaml
client.rest
clients
config
cspell.json
datasets
docker-compose.yml
docs
logs-docker
logs-dockerprocess-env-usages.txt
node_modules
package-lock.json
package.json
packages
process-env-migration-preview.txt
scripts
secrets
test-path-sanitization.mjs
tests
tools
tsconfig.base.json

=== Estrutura do DESTINO ===
.dockerignore
.editorconfig
.env.remote
.env.remote.example
.git
.github
.gitignore
.prettierrc
.venv
.vscode
Dockerfile
README.md
RUNBOOK_CLONE_REPOSITORIO.ipynb
RUNBOOK_CLONE_REPOSITORIO.md
apps
dist
docker-compose.remote.yml
eslint.config.mjs
include
lib
nest-cli.json
node_modules
package-lock.json
package.json
packages
scripts
spec
tools
tsconfig.bui

## 2. Fase 1 — Fundação: npm Workspaces + Shared Packages <a id='fase1'></a>

**Objetivo:** Configurar a base de compartilhamento de código entre os apps do monorepo.

**Artefatos a criar:**
- `packages/shared/` — Contratos e tipos compartilhados
- `packages/infra/` — Guards, interceptors, decorators compartilhados
- npm workspaces configurados no `package.json` raiz

In [6]:
import os

# Fase 1.1 — Verificar se npm workspaces já estão configurados

os.chdir(DESTINO)

print("=== package.json raiz (campo workspaces) ===")
subprocess.run(
    ["node", "-e", "const p = require('./package.json'); console.log('workspaces:', JSON.stringify(p.workspaces, null, 2))"],
    check=False,
)

=== package.json raiz (campo workspaces) ===
workspaces: [
  "apps/*",
  "packages/*"
]


CompletedProcess(args=['node', '-e', "const p = require('./package.json'); console.log('workspaces:', JSON.stringify(p.workspaces, null, 2))"], returncode=0)

## Fase 1.3 — Renomear pacote shared

Objetivo: validar o `package.json` de `packages/shared` e registrar a ação manual de renomear o campo `name` para `@monorepo-ai-llm/shared`.

In [ ]:
# O que faz: verifica package.json de packages/shared e orienta renomeacao manual do campo name
# Onde executa: shell local via bash -lc no diretorio do repositorio destino
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
SHARED_PKG="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/package.json"

# Ver package.json atual
cat "$SHARED_PKG"

echo ""
echo "ACAO MANUAL: editar $SHARED_PKG e trocar o campo name para @monorepo-ai-llm/shared"
''')

cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')

### Nota de execucao

Se o campo `name` ainda estiver com valor antigo, atualize manualmente o arquivo e so depois siga para a proxima celula.

In [14]:
from pathlib import Path
import shutil

# Fase 1.2 — Criar estrutura packages/shared (copiar da origem)
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

# Criar diretório packages/
packages_dir = Path(DESTINO) / "packages"
shared_src = Path(ORIGEM) / "packages" / "shared"
shared_dst = packages_dir / "shared"

# Criar diretório packages/
packages_dir.mkdir(parents=True, exist_ok=True)

# Copiar packages/shared da origem (idempotente)
if not shared_src.exists():
    raise FileNotFoundError(f"Origem não encontrada: {shared_src}")

shutil.copytree(shared_src, shared_dst, dirs_exist_ok=True)

print("=== Estrutura do packages/shared criado ===")
for f in sorted(shared_dst.rglob("*")):
    if f.is_file():
        print(f)
        # limitar em 30 arquivos
        if sum(1 for _ in sorted(shared_dst.rglob("*")) if _.is_file()) <= 30:
            continue

=== Estrutura do packages/shared criado ===
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/package.json
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/src/container-client.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/src/contracts/admin/api-key.contract.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/src/contracts/admin/index.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/src/contracts/common/api-response.contract.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/src/contracts/common/error-response.contract.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/src/contracts/common/pagination.contract.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/src/contracts/llm-ops/agent.contract.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/src/contracts/llm-ops/chat.contract.ts
/mnt/reposit

In [8]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 1.3 — Renomear pacote de @api-llm-embedded/shared → @monorepo-ai-llm/shared
SHARED_PKG="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/package.json"

# Ver package.json atual
cat "$SHARED_PKG"

# Substituir nome do pacote
# ATENÇÃO: execute manualmente via editor para garantir JSON válido
echo ""
echo "AÇÃO MANUAL: editar $SHARED_PKG e trocar o campo name para @monorepo-ai-llm/shared"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


{
  "name": "@api-llm-embedded/shared",
  "version": "1.0.0",
  "private": true,
  "type": "module",
  "types": "./src/index.ts",
  "exports": {
    ".": "./src/index.ts"
  },
  "sideEffects": false
}

AÇÃO MANUAL: editar /mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/shared/package.json e trocar o campo name para @monorepo-ai-llm/shared



In [9]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 1.4 — Criar packages/infra (extrair de apps/api/src/common/)
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

mkdir -p "$DESTINO/packages/infra/src"

# Copiar decorators, guards, filters, interceptors
for dir in decorators guards filters interceptors; do
  cp -r "$ORIGEM/apps/api/src/common/$dir" "$DESTINO/packages/infra/src/$dir"
  echo "Copiado: $dir"
done

echo ""
echo "=== Estrutura packages/infra ==="
find "$DESTINO/packages/infra" -type f
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


Copiado: decorators
Copiado: guards
Copiado: filters
Copiado: interceptors

=== Estrutura packages/infra ===
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/infra/src/decorators/current-user.decorator.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/infra/src/decorators/require-api-key.decorator.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/infra/src/decorators/skip-permission-validation.decorator.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/infra/src/filters/http-exception.filter.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/infra/src/guards/api-key-auth.guard.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/infra/src/guards/api-key-auth.service.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/infra/src/guards/guards.module.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/packages/infra/src/guards/permission-validation.guard.ts
/mnt/repositorio/workdir/reposto

In [10]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 1.5 — Verificar npm install após configurar workspaces
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

# Instalar dependências do workspace
npm install

echo "Código de saída: $?"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')



added 1 package, removed 1 package, and audited 751 packages in 2s

161 packages are looking for funding
  run `npm fund` for details

2 moderate severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
Código de saída: 0



In [11]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 1 — Validação: build ainda funciona após configurar workspaces
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

npm run build:users-api
echo "users-api build: $?"

npm run build:llm-ops-api
echo "llm-ops-api build: $?"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')



> monorepo-ai-llm@0.0.1 build:users-api
> nest build users-api

webpack 5.106.0 compiled successfully in 2128 ms
users-api build: 0

> monorepo-ai-llm@0.0.1 build:llm-ops-api
> nest build llm-ops-api

webpack 5.106.0 compiled successfully in 2020 ms
llm-ops-api build: 0



## 3. Fase 2 — Infraestrutura de Banco de Dados <a id='fase2'></a>

**Objetivo:** Configurar TypeORM, PostgreSQL e variáveis de ambiente.

**Artefatos a criar:**
- `docker-compose.yml` com postgres + langflow
- `.env.example` no root
- Módulo database em cada app que precisa de DB

In [15]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 2.1 — Instalar dependências de banco de dados
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

npm install @nestjs/typeorm typeorm pg @nestjs/config class-validator class-transformer zod
npm install --save-dev @types/pg

echo "Dependências instaladas: $?"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')



added 1 package, removed 1 package, and audited 758 packages in 2s

162 packages are looking for funding
  run `npm fund` for details

2 moderate severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.

up to date, audited 758 packages in 1s

162 packages are looking for funding
  run `npm fund` for details

2 moderate severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
Dependências instaladas: 0



In [16]:
# Fase 2.2 — Verificar docker-compose.remote.yml criado
compose_file = Path(DESTINO) / "docker-compose.remote.yml"

if compose_file.exists():
    print(f"=== {compose_file} ===")
    print(compose_file.read_text(encoding="utf-8"))
else:
    raise FileNotFoundError(f"Arquivo não encontrado: {compose_file}")

=== /mnt/repositorio/workdir/repostorios/monorepo-ai-llm/docker-compose.remote.yml ===
services:
  postgres:
    image: postgres:16-alpine
    container_name: monorepo-postgres
    environment:
      POSTGRES_DB: ${POSTGRES_DB:-monorepo_ai_llm}
      POSTGRES_USER: ${POSTGRES_USER:-monorepo}
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD:-monorepo123}
    ports:
      - "${POSTGRES_PORT:-5432}:5432"
    volumes:
      - postgres_data:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U ${POSTGRES_USER:-monorepo} -d ${POSTGRES_DB:-monorepo_ai_llm}"]
      interval: 10s
      timeout: 5s
      retries: 5
    restart: unless-stopped

  monorepo-ai-llm:
    build:
      context: .
      dockerfile: Dockerfile
      args:
        APP_NAME: monorepo-ai-llm
    image: monorepo-ai-llm/monorepo-ai-llm:remote
    container_name: monorepo-ai-llm
    environment:
      NODE_ENV: production
      PORT: 3000
      DATABASE_URL: ${DATABASE_URL:-postgresql://monorepo:monore

In [18]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 2.3 — Subir postgres para testes locais
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

docker compose up -d postgres
echo "PostgreSQL status: $?"

# Aguardar postgres ficar pronto
sleep 3
docker compose ps
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    if "no configuration file provided" in run.stderr:
        print("Tentando novamente com docker-compose.remote.yml...")
        retry = subprocess.run(
            [
                "bash",
                "-lc",
                f'cd "{DESTINO}" && docker compose -f docker-compose.remote.yml up -d postgres && sleep 3 && docker compose -f docker-compose.remote.yml ps',
            ],
            text=True,
            capture_output=True,
        )
        print(retry.stdout)
        if retry.returncode != 0:
            print(retry.stderr)
            raise RuntimeError(f'Falha (exit={retry.returncode}) nesta celula')
    else:
        raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


PostgreSQL status: 1

no configuration file provided: not found
no configuration file provided: not found

Tentando novamente com docker-compose.remote.yml...
NAME                IMAGE                                    COMMAND                  SERVICE           CREATED         STATUS                            PORTS
llm-ops-api         monorepo-ai-llm/llm-ops-api:remote       "docker-entrypoint.s…"   llm-ops-api       6 hours ago     Up 6 hours                        0.0.0.0:3002->3002/tcp, [::]:3002->3002/tcp
monorepo-ai-llm     monorepo-ai-llm/monorepo-ai-llm:remote   "docker-entrypoint.s…"   monorepo-ai-llm   6 hours ago     Up 6 hours                        0.0.0.0:3000->3000/tcp, [::]:3000->3000/tcp
monorepo-postgres   postgres:16-alpine                       "docker-entrypoint.s…"   postgres          4 seconds ago   Up 3 seconds (health: starting)   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp
sharepoint-api      monorepo-ai-llm/sharepoint-api:remote    "docker-entrypoint.s…"   s

In [32]:
# O que faz: copia infra de database e aplica ajustes automáticos no typeorm.config.ts, registrando alterações
# Onde executa: shell local via bash -lc com cwd no repositório destino
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/users-api/src/infra/database"

# Copiar arquivos base de database
cp "$ORIGEM/apps/api/src/infra/database/database.module.ts" "$DESTINO/apps/users-api/src/infra/database/"
cp "$ORIGEM/apps/api/src/infra/database/typeorm.config.ts" "$DESTINO/apps/users-api/src/infra/database/"

# Copiar migrations relevantes para users-api
mkdir -p "$DESTINO/apps/users-api/src/infra/database/migrations"
for mig in 1713830500000-CreateUsersTable.ts 1713830600000-CreateSyncSchema.ts 1777593600000-CreateApiKeysTable.ts; do
  if [ -f "$ORIGEM/apps/api/src/infra/database/migrations/$mig" ]; then
    cp "$ORIGEM/apps/api/src/infra/database/migrations/$mig" "$DESTINO/apps/users-api/src/infra/database/migrations/$mig"
    echo "✓ Migration copiada: $mig"
  fi
done

TARGET_CONFIG="$DESTINO/apps/users-api/src/infra/database/typeorm.config.ts"

# Ajustes automáticos no arquivo copiado + registro do que foi alterado
python - <<'PY'
from pathlib import Path
from datetime import datetime

cfg = Path('/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/infra/database/typeorm.config.ts')
text = cfg.read_text(encoding='utf-8')
updated = text
changes = []

# 1) Import ESM para NodeNext
old = "from '../../modules/llm-ops/entities/index';"
new = "from '../../modules/llm-ops/entities/index.js';"
if old in updated:
    updated = updated.replace(old, new)
    changes.append("import entities index -> index.js")

# 2) Padronizar package name compartilhado
old_pkg = "@api-llm-embedded/shared"
new_pkg = "@monorepo-ai-llm/shared"
if old_pkg in updated:
    updated = updated.replace(old_pkg, new_pkg)
    changes.append("package shared rename")

# 3) Garantir fallback de schema public para PG_SCHEMA
if "schema: process.env.PG_SCHEMA ?? 'public'" not in updated and "schema: process.env.PG_SCHEMA" in updated:
    updated = updated.replace("schema: process.env.PG_SCHEMA", "schema: process.env.PG_SCHEMA ?? 'public'")
    changes.append("PG_SCHEMA fallback public")

cfg.write_text(updated, encoding='utf-8')

log_dir = Path('/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/spec/execucao/logs')
log_dir.mkdir(parents=True, exist_ok=True)
log_file = log_dir / 'fase09-fase24-autofix.log'

lines = [
    f"[{datetime.now().isoformat(timespec='seconds')}] fase 2.4 autoajuste",
    f"arquivo: {cfg}",
]
if changes:
    lines.extend([f"- {c}" for c in changes])
else:
    lines.append("- sem mudanças textuais necessárias")

with log_file.open('a', encoding='utf-8') as fp:
    fp.write("\n".join(lines) + "\n\n")

print("=== Ajustes no typeorm.config.ts ===")
if changes:
    for c in changes:
        print(f"- {c}")
else:
    print("- sem mudanças textuais necessárias")
print(f"Log: {log_file}")
PY

echo ""
echo "=== Migrations em users-api (destino) ==="
find "$DESTINO/apps/users-api/src/infra/database/migrations" -type f | sort

echo ""
echo "=== Arquivo final: typeorm.config.ts (trecho inicial) ==="
sed -n '1,80p' "$TARGET_CONFIG"
''')

cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')

✓ Migration copiada: 1713830500000-CreateUsersTable.ts
✓ Migration copiada: 1713830600000-CreateSyncSchema.ts
✓ Migration copiada: 1777593600000-CreateApiKeysTable.ts

=== Migrations em users-api (destino) ===
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/infra/database/migrations/1713830500000-CreateUsersTable.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/infra/database/migrations/1713830600000-CreateSyncSchema.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/infra/database/migrations/1777593600000-CreateApiKeysTable.ts

=== Arquivo final: typeorm.config.ts (trecho inicial) ===
import { DataSource, type DataSourceOptions } from 'typeorm';
import { ConfigService } from '@nestjs/config';
import type { TypeOrmModuleOptions } from '@nestjs/typeorm';
import { dirname, resolve } from 'node:path';
import { fileURLToPath } from 'node:url';
import {
  InteractionLearningEventEntity,
  LlmOpsAgentEntity,
  PromptDe

### Fase 2.5 — Entities de Users e API Keys (onde estão e como copiar)

Observação importante: a célula 2.4 trata **infra de banco** (`database.module`, `typeorm.config`, migrations). As entities de domínio ficam em `apps/api/src/modules/*/entities` na origem.

No projeto de origem, para o escopo de users-api, os caminhos principais são:
- `apps/api/src/modules/users/entities/user.entity.ts`
- `apps/api/src/modules/api-keys/entities/api-key.entity.ts`

`admin` e `governance` podem não ter entities próprias (dependendo da modelagem por serviço/DTO).

In [33]:
# O que faz: localiza e copia entities de users/api-keys da origem para users-api no destino
# Onde executa: shell local via bash -lc com cwd no repositorio destino
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

echo "=== Entities na origem (users/api-keys/admin/governance) ==="
for mod in users api-keys admin governance; do
  echo "-- $mod --"
  find "$ORIGEM/apps/api/src/modules/$mod" -type f -name '*entity.ts' 2>/dev/null || true
  echo ""
done

mkdir -p "$DESTINO/apps/users-api/src/modules/users/entities"
mkdir -p "$DESTINO/apps/users-api/src/modules/api-keys/entities"

if [ -f "$ORIGEM/apps/api/src/modules/users/entities/user.entity.ts" ]; then
  cp "$ORIGEM/apps/api/src/modules/users/entities/user.entity.ts" \
     "$DESTINO/apps/users-api/src/modules/users/entities/"
  echo "✓ Copiado: user.entity.ts"
else
  echo "⚠ user.entity.ts nao encontrado na origem"
fi

if [ -f "$ORIGEM/apps/api/src/modules/api-keys/entities/api-key.entity.ts" ]; then
  cp "$ORIGEM/apps/api/src/modules/api-keys/entities/api-key.entity.ts" \
     "$DESTINO/apps/users-api/src/modules/api-keys/entities/"
  echo "✓ Copiado: api-key.entity.ts"
else
  echo "⚠ api-key.entity.ts nao encontrado na origem"
fi

echo ""
echo "=== Entities no destino (users-api) ==="
find "$DESTINO/apps/users-api/src/modules" -type f -name '*entity.ts' 2>/dev/null || true
''')

cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')

=== Entities na origem (users/api-keys/admin/governance) ===
-- users --
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/users/entities/user.entity.ts

-- api-keys --
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/api-keys/entities/api-key.entity.ts

-- admin --

-- governance --

✓ Copiado: user.entity.ts
✓ Copiado: api-key.entity.ts

=== Entities no destino (users-api) ===
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/modules/api-keys/api-keys/entities/api-key.entity.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/modules/api-keys/entities/api-key.entity.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/modules/users/entities/user.entity.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/modules/users/users/entities/user.entity.ts



## 4. Fase 3 — Migração: users-api <a id='fase3'></a>

**Objetivo:** Migrar módulos `users`, `admin`, `governance`, `api-keys` para `apps/users-api`.

**Schema PostgreSQL:** `public`  
**Porta:** 3001

In [34]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 3.1 — Listar módulos a migrar na origem
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"

for mod in users admin governance api-keys health; do
  echo "=== Módulo: $mod ==="
  find "$ORIGEM/apps/api/src/modules/$mod" -type f -name '*.ts' | head -10
  echo ""
done
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


=== Módulo: users ===
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/users/dto/create-user.dto.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/users/dto/update-user.dto.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/users/entities/user.entity.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/users/types/user-role.type.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/users/users.controller.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/users/users.module.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/users/users.service.ts

=== Módulo: admin ===
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/admin/admin.controller.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/admin/admin.module.ts
/mnt/repositorio/workdir/repostorios/api-ll

In [35]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 3.2 — Copiar módulos para users-api
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/users-api/src/modules"

for mod in users admin governance api-keys health; do
  if [ -d "$ORIGEM/apps/api/src/modules/$mod" ]; then
    cp -r "$ORIGEM/apps/api/src/modules/$mod" "$DESTINO/apps/users-api/src/modules/$mod"
    echo "✓ Copiado: $mod"
  else
    echo "✗ Não encontrado: $mod"
  fi
done
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


✓ Copiado: users
✓ Copiado: admin
✓ Copiado: governance
✓ Copiado: api-keys
✓ Copiado: health



In [36]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 3.3 — Identificar imports de @api-llm-embedded/shared que precisam ser atualizados
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

echo "=== Arquivos com @api-llm-embedded/shared em users-api ==="
grep -rl '@api-llm-embedded/shared' "$DESTINO/apps/users-api/src/" 2>/dev/null | head -20

echo ""
echo "AÇÃO: substituir @api-llm-embedded/shared por @monorepo-ai-llm/shared nos arquivos acima"
echo "Comando: find ... -exec sed -i 's/@api-llm-embedded\/shared/@monorepo-ai-llm\/shared/g' {} +"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


=== Arquivos com @api-llm-embedded/shared em users-api ===
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/modules/users/users/dto/create-user.dto.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/modules/users/users/dto/update-user.dto.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/modules/users/users/users.controller.ts
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/users-api/src/modules/users/users/users.service.ts

AÇÃO: substituir @api-llm-embedded/shared por @monorepo-ai-llm/shared nos arquivos acima
Comando: find ... -exec sed -i 's/@api-llm-embedded\/shared/@monorepo-ai-llm\/shared/g' {} +



In [39]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 3.4 — Tentar build de users-api após migração
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

npm run build:users-api 2>&1 | tail -30
echo "Build exit code: $?"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')



> monorepo-ai-llm@0.0.1 build:users-api
> nest build users-api

webpack 5.106.0 compiled successfully in 2410 ms
Build exit code: 0



In [40]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 3.5 — Smoke test users-api (requer PostgreSQL rodando)
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

# Iniciar app em background e testar
nest start users-api &
APP_PID=$!

sleep 5

# Teste básico de health
curl -s http://localhost:3001/health
echo ""

# Encerrar app
kill $APP_PID 2>/dev/null
echo "Smoke test concluído"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


{"status":"ok"}
Smoke test concluído



## 5. Fase 4 — Migração: llm-ops-api <a id='fase4'></a>

**Objetivo:** Migrar módulo `llm-ops` (10 entities, RAG, AstraDB, LangFlow) e `audit` para `apps/llm-ops-api`.

**Schema PostgreSQL:** `llm_ops`  
**Porta:** 3002

**Integração RAG:**
```
llm-ops-api → AstraDB (knowledge_base/interactions) → LangFlow (context-only)
Flow ID: f81c0124-ffc2-4458-b30d-4d588d393518
```

In [41]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 4.1 — Listar entidades do domínio llm-ops na origem
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"

echo "=== Entidades em llm-ops ==="
find "$ORIGEM/apps/api/src/modules/llm-ops" -name '*.entity.ts' -type f

echo ""
echo "=== DTOs em llm-ops ==="
find "$ORIGEM/apps/api/src/modules/llm-ops" -name '*.dto.ts' -type f | wc -l
echo "DTOs encontrados"

echo ""
echo "=== AstraDB client ==="
ls "$ORIGEM/apps/api/src/infra/astradb/" 2>/dev/null || echo "(verificar caminho)"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


=== Entidades em llm-ops ===
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/llm-ops/entities/interaction-learning-event.entity.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/llm-ops/entities/llm-ops-agent.entity.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/llm-ops/entities/prompt-dependency.entity.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/llm-ops/entities/prompt-template.entity.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/llm-ops/entities/prompt-usage-history.entity.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/llm-ops/entities/prompt-validation.entity.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/llm-ops/entities/prompt-version.entity.ts
/mnt/repositorio/workdir/repostorios/api-llm-embedded/apps/api/src/modules/llm-ops/entities/topic-flow-version.entity.ts
/mnt/re

In [42]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 4.2 — Copiar módulos para llm-ops-api
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/llm-ops-api/src/modules"
mkdir -p "$DESTINO/apps/llm-ops-api/src/infra"

# Módulos de negócio
for mod in llm-ops audit health; do
  if [ -d "$ORIGEM/apps/api/src/modules/$mod" ]; then
    cp -r "$ORIGEM/apps/api/src/modules/$mod" "$DESTINO/apps/llm-ops-api/src/modules/$mod"
    echo "✓ Copiado módulo: $mod"
  fi
done

# Infra: AstraDB
if [ -d "$ORIGEM/apps/api/src/infra/astradb" ]; then
  cp -r "$ORIGEM/apps/api/src/infra/astradb" "$DESTINO/apps/llm-ops-api/src/infra/astradb"
  echo "✓ Copiado: infra/astradb"
fi

# RAG LangFlow
LANGFLOW_SRC="$ORIGEM/apps/svcia/mcp-llm-ops/src/rag-langflow"
if [ -d "$LANGFLOW_SRC" ]; then
  mkdir -p "$DESTINO/apps/llm-ops-api/src/rag-langflow"
  cp -r "$LANGFLOW_SRC/" "$DESTINO/apps/llm-ops-api/src/rag-langflow/"
  echo "✓ Copiado: rag-langflow"
fi
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


✓ Copiado módulo: llm-ops
✓ Copiado módulo: audit
✓ Copiado módulo: health
✓ Copiado: rag-langflow



In [1]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 4.3 — Build de llm-ops-api
set -o pipefail
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

npm run build:llm-ops-api 2>&1 | tail -30
BUILD_EXIT=${PIPESTATUS[0]}
echo "Build exit code: $BUILD_EXIT"
exit $BUILD_EXIT
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')



> monorepo-ai-llm@0.0.1 build:llm-ops-api
> nest build llm-ops-api

webpack 5.106.0 compiled successfully in 2959 ms
Build exit code: 0



## 6. Fase 5 — Migração: sharepoint-api <a id='fase5'></a>

**Objetivo:** Migrar `sharepoint` e `graph` (Microsoft Graph) para `apps/sharepoint-api`.

**App stateless** — sem banco de dados.  
**Porta:** 3003  
**Auth:** Microsoft Entra ID + OAuth + certificado

In [2]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 5.1 — Copiar módulos para sharepoint-api
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/sharepoint-api/src/modules"

for mod in sharepoint graph health; do
  if [ -d "$ORIGEM/apps/api/src/modules/$mod" ]; then
    cp -r "$ORIGEM/apps/api/src/modules/$mod" "$DESTINO/apps/sharepoint-api/src/modules/$mod"
    echo "✓ Copiado: $mod"
  fi
done

echo ""
echo "ATENÇÃO: sharepoint-api é STATELESS — não incluir DatabaseModule"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


✓ Copiado: sharepoint
✓ Copiado: graph
✓ Copiado: health

ATENÇÃO: sharepoint-api é STATELESS — não incluir DatabaseModule



In [4]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 5.2 — Build de sharepoint-api
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

npm run build:sharepoint-api 2>&1 | tail -30
echo "Build exit code: $?"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')



> monorepo-ai-llm@0.0.1 build:sharepoint-api
> nest build sharepoint-api

webpack 5.106.0 compiled successfully in 2703 ms
Build exit code: 0



## 7. Fase 6 — Migração: sync-api <a id='fase6'></a>

**Objetivo:** Migrar `sync` e `m365-migration` (8 entities) para `apps/sync-api`.

**Schema PostgreSQL:** `public`  
**Porta:** 3004

In [7]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 6.1 — Copiar módulos para sync-api
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/sync-api/src/modules"

for mod in sync m365-migration health graph; do
  if [ -d "$ORIGEM/apps/api/src/modules/$mod" ]; then
    cp -r "$ORIGEM/apps/api/src/modules/$mod" "$DESTINO/apps/sync-api/src/modules/$mod"
    echo "✓ Copiado: $mod"
  fi
done
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


✓ Copiado: sync
✓ Copiado: m365-migration
✓ Copiado: health
✓ Copiado: graph



In [12]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 6.2 — Build de sync-api
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

npm run build:sync-api 2>&1 | tail -30
echo "Build exit code: $?"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')



> monorepo-ai-llm@0.0.1 build:sync-api
> nest build sync-api

webpack 5.106.0 compiled successfully in 3107 ms
Build exit code: 0



## 8. Fase 7 — Migração: MCP Servers <a id='fase7'></a>

**Objetivo:** Migrar os 5 servidores MCP de `apps/svcia/` e o `tools/mcp-project-health`.

**Todos os MCP servers são read-only** — nenhuma mutação via MCP.

| Server | Origem | Destino |
|---|---|---|
| mcp-users | `apps/svcia/mcp-users/` | `apps/svcia/mcp-users/` |
| mcp-llm-ops | `apps/svcia/mcp-llm-ops/` | `apps/svcia/mcp-llm-ops/` |
| mcp-sync | `apps/svcia/mcp-sync/` | `apps/svcia/mcp-sync/` |
| mcp-secrets | `apps/svcia/mcp-secrets/` | `apps/svcia/mcp-secrets/` |
| mcp-awesome-copilot | `apps/mcp-awesome-copilot/` | `apps/svcia/mcp-awesome-copilot/` |
| mcp-project-health | `tools/mcp-project-health/` | `tools/mcp-project-health/` |

In [13]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 7.1 — Copiar apps/svcia/ para o monorepo
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

mkdir -p "$DESTINO/apps/svcia"

# Copiar cada MCP server
for server in mcp-users mcp-llm-ops mcp-sync mcp-secrets; do
  if [ -d "$ORIGEM/apps/svcia/$server" ]; then
    cp -r "$ORIGEM/apps/svcia/$server" "$DESTINO/apps/svcia/$server"
    echo "✓ Copiado: $server"
  fi
done

# mcp-awesome-copilot tem localização diferente na origem
if [ -d "$ORIGEM/apps/mcp-awesome-copilot" ]; then
  cp -r "$ORIGEM/apps/mcp-awesome-copilot" "$DESTINO/apps/svcia/mcp-awesome-copilot"
  echo "✓ Copiado: mcp-awesome-copilot"
fi

echo ""
echo "=== Estrutura apps/svcia no destino ==="
ls "$DESTINO/apps/svcia/"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


✓ Copiado: mcp-users
✓ Copiado: mcp-llm-ops
✓ Copiado: mcp-sync
✓ Copiado: mcp-secrets

=== Estrutura apps/svcia no destino ===
mcp-llm-ops
mcp-secrets
mcp-sync
mcp-users



In [14]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 7.2 — Copiar tools/mcp-project-health
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

cp -r "$ORIGEM/tools/mcp-project-health" "$DESTINO/tools/mcp-project-health"
echo "✓ Copiado: tools/mcp-project-health"

echo ""
echo "=== Estrutura tools/ no destino ==="
ls "$DESTINO/tools/"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


✓ Copiado: tools/mcp-project-health

=== Estrutura tools/ no destino ===
mcp
mcp-project-health



In [15]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 7.3 — Renomear packages em todos os MCP servers
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

echo "=== package.json dos MCP servers (campo name) ==="
find "$DESTINO/apps/svcia" -name 'package.json' -maxdepth 3 | while read f; do
  NAME=$(node -e "console.log(require('$f').name)" 2>/dev/null)
  echo "$f: $NAME"
done

echo ""
echo "AÇÃO: substituir @api-llm-embedded/ por @monorepo-ai-llm/ em cada package.json"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


=== package.json dos MCP servers (campo name) ===
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/svcia/mcp-llm-ops/package.json: @api-llm-embedded/mcp-llm-ops
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/svcia/mcp-secrets/package.json: @api-llm-embedded/mcp-secrets
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/svcia/mcp-sync/package.json: @api-llm-embedded/mcp-sync
/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/apps/svcia/mcp-users/package.json: @api-llm-embedded/mcp-users

AÇÃO: substituir @api-llm-embedded/ por @monorepo-ai-llm/ em cada package.json



In [16]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 7.4 — Build dos MCP servers (cada um usa seu próprio build script)
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

for server in mcp-users mcp-llm-ops mcp-sync mcp-secrets; do
  echo "=== Build: $server ==="
  cd "$DESTINO/apps/svcia/$server"
  npm run build 2>&1 | tail -5
  echo "Exit code: $?"
  echo ""
done
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


=== Build: mcp-users ===

> @api-llm-embedded/mcp-users@1.0.0 build
> tsgo -p tsconfig.json

sh: 1: tsgo: not found
Exit code: 0

=== Build: mcp-llm-ops ===

> @api-llm-embedded/mcp-llm-ops@1.0.0 build
> tsgo -p tsconfig.json

sh: 1: tsgo: not found
Exit code: 0

=== Build: mcp-sync ===

> @api-llm-embedded/mcp-sync@1.0.0 build
> tsgo -p tsconfig.json

sh: 1: tsgo: not found
Exit code: 0

=== Build: mcp-secrets ===

> @api-llm-embedded/mcp-secrets@1.0.0 build
> tsgo -p tsconfig.json

sh: 1: tsgo: not found
Exit code: 0




In [17]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 7.5 — Smoke test dos MCP servers (requer apis rodando)
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

# Testar mcp-users (requer users-api em :3001)
if [ -f "$DESTINO/scripts/smoke-tests/mcp-users-read-smoke.mjs" ]; then
  node "$DESTINO/scripts/smoke-tests/mcp-users-read-smoke.mjs"
else
  echo "Smoke test ainda não migrado para o destino — executar Fase 8 primeiro"
fi
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


Smoke test ainda não migrado para o destino — executar Fase 8 primeiro



## 9. Fase 8 — Scripts, Config e Docs <a id='fase8'></a>

**Objetivo:** Migrar scripts de smoke tests, configurações MCP e documentação de arquitetura.

In [18]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 8.1 — Copiar scripts/
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

mkdir -p "$DESTINO/scripts"

# Copiar smoke tests
cp -r "$ORIGEM/scripts/smoke-tests" "$DESTINO/scripts/smoke-tests"
echo "✓ Copiado: scripts/smoke-tests"

# Copiar bootstrap scripts
cp -r "$ORIGEM/scripts/bootstrap" "$DESTINO/scripts/bootstrap" 2>/dev/null && echo "✓ Copiado: scripts/bootstrap"

echo ""
echo "AÇÃO: atualizar caminhos hardcoded que referenciam 'api-llm-embedded' nos .mjs"
grep -r 'api-llm-embedded' "$DESTINO/scripts/" 2>/dev/null | head -10
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


✓ Copiado: scripts/smoke-tests
✓ Copiado: scripts/bootstrap

AÇÃO: atualizar caminhos hardcoded que referenciam 'api-llm-embedded' nos .mjs



In [19]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 8.2 — Copiar config/mcp (templates de configuração)
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

mkdir -p "$DESTINO/config"
cp -r "$ORIGEM/config/mcp" "$DESTINO/config/mcp"
echo "✓ Copiado: config/mcp"

echo ""
echo "=== Config MCP copiados ==="
ls "$DESTINO/config/mcp/"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


✓ Copiado: config/mcp

=== Config MCP copiados ===
langflow-streamable.example.json
shp-local-mcp.example.json



In [20]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 8.3 — Copiar docs de arquitetura
ORIGEM="/mnt/repositorio/workdir/repostorios/api-llm-embedded"
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

if [ -d "$ORIGEM/docs" ]; then
  mkdir -p "$DESTINO/docs"
  cp -r "$ORIGEM/docs/" "$DESTINO/docs/"
  echo "✓ Copiado: docs/"
  ls "$DESTINO/docs/"
fi
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


✓ Copiado: docs/
docs



## 10. Fase 9 — CLAUDE.md <a id='fase9'></a>

**Objetivo:** Criar CLAUDE.md no root do monorepo baseado no projeto origem, adaptado para a nova estrutura.

O `CLAUDE.md` é lido automaticamente pelo Claude Code em cada sessão. Ele deve refletir a arquitetura **atual** do monorepo após a migração.

In [21]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 9.1 — Verificar se CLAUDE.md existe no destino
DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"

if [ -f "$DESTINO/CLAUDE.md" ]; then
  echo "CLAUDE.md já existe. Tamanho:"
  wc -l "$DESTINO/CLAUDE.md"
else
  echo "CLAUDE.md NÃO existe — precisa ser criado"
  echo "Referência: /mnt/repositorio/workdir/repostorios/api-llm-embedded/CLAUDE.md"
fi
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


CLAUDE.md NÃO existe — precisa ser criado
Referência: /mnt/repositorio/workdir/repostorios/api-llm-embedded/CLAUDE.md



In [22]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 9.2 — Criar CLAUDE.md baseado na origem
# AÇÃO MANUAL: criar ou editar CLAUDE.md no destino
# Pontos-chave a adaptar:

echo "Checklist CLAUDE.md:"
echo "[ ] Atualizar nomes de pacotes: @api-llm-embedded/ → @monorepo-ai-llm/"
echo "[ ] Atualizar estrutura de apps (5 NestJS apps separados)"
echo "[ ] Atualizar comandos de build (NestJS CLI em vez de tsgo direto)"
echo "[ ] Atualizar caminhos de MCP servers (apps/svcia/)"
echo "[ ] Remover referências a tsgo como compilador principal"
echo "[ ] Manter seções de regras de governance e segurança"
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


Checklist CLAUDE.md:
[ ] Atualizar nomes de pacotes: @api-llm-embedded/ → @monorepo-ai-llm/
[ ] Atualizar estrutura de apps (5 NestJS apps separados)
[ ] Atualizar comandos de build (NestJS CLI em vez de tsgo direto)
[ ] Atualizar caminhos de MCP servers (apps/svcia/)
[ ] Remover referências a tsgo como compilador principal
[ ] Manter seções de regras de governance e segurança



## 11. Fase 10 — Validação Final <a id='fase10'></a>

**Objetivo:** Garantir que o monorepo migrado está funcional em todos os aspectos.

In [23]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 10.1 — Build completo
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm
set -o pipefail

echo "=== Build de todos os apps ==="
BUILD_FAILED=0
for app in users-api llm-ops-api sharepoint-api sync-api; do
  echo "Building $app..."
  npm run build:$app 2>&1 | tail -3
  BUILD_EXIT=$?
  echo "Exit: $BUILD_EXIT"
  if [ "$BUILD_EXIT" -ne 0 ]; then
    BUILD_FAILED=$BUILD_EXIT
  fi
  echo ""
done
exit $BUILD_FAILED
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


=== Build de todos os apps ===
Building users-api...
> nest build users-api

webpack 5.106.0 compiled successfully in 2383 ms
Exit: 0

Building llm-ops-api...
> nest build llm-ops-api

webpack 5.106.0 compiled successfully in 2923 ms
Exit: 0

Building sharepoint-api...
> nest build sharepoint-api

webpack 5.106.0 compiled successfully in 2553 ms
Exit: 0

Building sync-api...
> nest build sync-api

webpack 5.106.0 compiled successfully in 3017 ms
Exit: 0




### Ajustes de lint/typecheck incorporados

Antes de executar as validacoes abaixo, o codigo do `llm-ops-api` foi estabilizado para o lint type-aware:

- `typeorm.config.ts`: `buildBaseOptions()` deixou de retornar `any` e passou a retornar `TypeOrmModuleOptions`.
- `main.ts`: `bootstrap()` foi marcado com `void` para satisfazer `no-floating-promises`.
- `audit.controller.ts`: removido cast `as any` no filtro `eventType`.
- `astra-rag.service.ts`: `recordInteraction()` deixou de ser `async` porque nao usa `await`.
- `llm-ops.service.ts`: removidos imports nao usados, casts `any` em `eventType`, comparacoes enum inseguras e `await` sobre metodo sincrono.
- Arquivos tocados foram formatados com Prettier.


In [31]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 10.2 — Type check
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

npx tsc --noEmit -p apps/llm-ops-api/tsconfig.app.json
TYPECHECK_EXIT=$?
echo "Type check exit code: $TYPECHECK_EXIT"
exit $TYPECHECK_EXIT
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


Type check exit code: 0



In [32]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 10.3 — Lint
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm

npx eslint "apps/llm-ops-api/src/**/*.ts" --ignore-pattern "**/*.spec.ts"
LINT_EXIT=$?
echo "Lint exit code: $LINT_EXIT"
exit $LINT_EXIT
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


Lint exit code: 0



In [35]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 10.4 — Smoke tests (requer PostgreSQL rodando)
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm
set -e -o pipefail

# Verificar se postgres está rodando
docker compose -f docker-compose.remote.yml --env-file .env.remote.example ps postgres

# Rodar smoke tests se disponíveis
if [ -f "scripts/smoke-tests/run-all.mjs" ]; then
  node scripts/smoke-tests/run-all.mjs
else
  echo "Smoke test runner não migrado ainda — executar individualmente"
  ls scripts/smoke-tests/ | head -10
fi
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


NAME                IMAGE                COMMAND                  SERVICE    CREATED       STATUS                 PORTS
monorepo-postgres   postgres:16-alpine   "docker-entrypoint.s…"   postgres   2 hours ago   Up 2 hours (healthy)   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp
Smoke test runner não migrado ainda — executar individualmente
smoke-tests



In [41]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 10.5 — Docker build validation
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm
set -o pipefail

docker compose -f docker-compose.remote.yml build --no-cache 2>&1 | tail -30
DOCKER_BUILD_EXIT=$?
echo "Docker build exit code: $DOCKER_BUILD_EXIT"
exit $DOCKER_BUILD_EXIT
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


#43 ...

#46 [sharepoint-api] resolving provenance for metadata file
#46 DONE 0.0s

#41 [users-api] exporting to image
#41 unpacking to docker.io/monorepo-ai-llm/users-api:remote 3.6s done
#41 DONE 9.7s

#43 [llm-ops-api] exporting to image
#43 unpacking to docker.io/monorepo-ai-llm/llm-ops-api:remote 3.3s done
#43 DONE 8.5s

#45 [sync-api] exporting to image
#45 unpacking to docker.io/monorepo-ai-llm/sync-api:remote 3.1s done
#45 DONE 8.2s

#47 [users-api] resolving provenance for metadata file
#47 DONE 0.1s

#48 [llm-ops-api] resolving provenance for metadata file
#48 DONE 0.1s

#49 [sync-api] resolving provenance for metadata file
#49 DONE 0.0s
 Image monorepo-ai-llm/llm-ops-api:remote Built 
 Image monorepo-ai-llm/monorepo-ai-llm:remote Built 
 Image monorepo-ai-llm/sharepoint-api:remote Built 
 Image monorepo-ai-llm/sync-api:remote Built 
 Image monorepo-ai-llm/users-api:remote Built 
Docker build exit code: 0



In [43]:
# O que faz: executa comandos shell da fase via bash -lc
# Onde executa: shell local com cwd em DESTINO (quando definido) ou cwd atual
import os
import subprocess
import textwrap

script = textwrap.dedent(r'''\
# Fase 10.6 — Checklist final de validacao
echo "=== CHECKLIST FINAL DE MIGRACAO ==="
echo ""

DESTINO="/mnt/repositorio/workdir/repostorios/monorepo-ai-llm"
CHECK_FAILED=0

# Verificar artefatos obrigatorios
check_exists() {
  if [ -e "$DESTINO/$1" ]; then
    echo "[OK] $1"
  else
    echo "[FALTA] $1"
    CHECK_FAILED=1
  fi
}

check_exists "packages/shared/src/index.ts"
check_exists "packages/infra/src/index.ts"
check_exists "apps/users-api/src/modules/users"
check_exists "apps/llm-ops-api/src/modules/llm-ops"
check_exists "apps/sharepoint-api/src/modules/sharepoint"
check_exists "apps/sync-api/src/modules/sync"
check_exists "apps/svcia/mcp-users"
check_exists "apps/svcia/mcp-llm-ops"
check_exists "apps/svcia/mcp-sync"
check_exists "apps/svcia/mcp-secrets"
check_exists "tools/mcp-project-health"
check_exists "scripts/smoke-tests"
check_exists "config/mcp"
check_exists "docker-compose.remote.yml"
check_exists ".env.example"
check_exists "CLAUDE.md"
check_exists "spec/fases/fase-09-plano-migracao-api-llm-embedded.md"
check_exists "spec/execucao/fase-09-migracao-api-llm-embedded.ipynb"

exit $CHECK_FAILED
''')
cwd = globals().get('DESTINO', os.getcwd())
run = subprocess.run(['bash', '-lc', script], cwd=cwd, text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Falha (exit={run.returncode}) nesta celula')


=== CHECKLIST FINAL DE MIGRACAO ===

[OK] packages/shared/src/index.ts
[OK] packages/infra/src/index.ts
[OK] apps/users-api/src/modules/users
[OK] apps/llm-ops-api/src/modules/llm-ops
[OK] apps/sharepoint-api/src/modules/sharepoint
[OK] apps/sync-api/src/modules/sync
[OK] apps/svcia/mcp-users
[OK] apps/svcia/mcp-llm-ops
[OK] apps/svcia/mcp-sync
[OK] apps/svcia/mcp-secrets
[OK] tools/mcp-project-health
[OK] scripts/smoke-tests
[OK] config/mcp
[OK] docker-compose.remote.yml
[OK] .env.example
[OK] CLAUDE.md
[OK] spec/fases/fase-09-plano-migracao-api-llm-embedded.md
[OK] spec/execucao/fase-09-migracao-api-llm-embedded.ipynb



## Conclusao

A migracao completa do `api-llm-embedded` para o `monorepo-ai-llm` e concluida quando todos os itens do checklist final estao marcados e os builds passam sem erros.

### Proximos passos apos a migracao

1. Atualizar configuracoes de MCP no VS Code para apontar para os novos caminhos
2. Configurar GitHub Actions para CI dos novos apps
3. Criar PR documentado com evidencias de validacao
4. Arquivar ou deprecar o repositorio `api-llm-embedded`

### Artefatos desta migracao

- `spec/fases/fase-09-plano-migracao-api-llm-embedded.md` — Plano detalhado
- `spec/execucao/fase-09-migracao-api-llm-embedded.ipynb` — Este notebook

---

*Conforme regra obrigatoria do README: toda migracao deve incluir um `.md` e um `.ipynb`.*

## Registro de estabilizacao TS6, Auth e PostgreSQL

### Contexto
Na evolucao do plano, foi aplicada a migracao para TypeScript 6.0.3 com monorepo NestJS em workspaces, pacote de autenticacao compartilhado em `packages/auth` e banco PostgreSQL em container.

### Erros encontrados
- `TS5101` por deprecacao de `baseUrl` no TS6.
- `TS5011` por `rootDir` nao explicito nos `tsconfig.app.json`.
- `TS6059` por import de `packages/auth` fora do escopo de `rootDir` das apps.
- `TS2307` em imports relativos ESM sem extensao no pacote `auth`.

### Correcoes aplicadas
- `tsconfig.json`: adicionado `ignoreDeprecations: "6.0"`.
- `apps/*/tsconfig.app.json`: adicionado `rootDir: "../../"`.
- `packages/auth/src/*.ts`: imports relativos padronizados com extensao `.js` para `NodeNext`.
- `nest-cli.json`: registro das libraries `shared` e `auth`.
- `docker-compose.remote.yml`: adicionado service `postgres` com healthcheck e `DATABASE_URL`.
- Autenticacao transversal aplicada com `AuthModule` + `Public()` em health.

### Resultado
Build de todas as apps concluido com sucesso apos os ajustes estruturais.

In [45]:
# O que faz: valida a compilacao das apps apos estabilizacao TS6/ESM/rootDir
# Onde executa: raiz do repositorio destino
import subprocess

DESTINO = '/mnt/repositorio/workdir/repostorios/monorepo-ai-llm'

commands = [
    ['npm', 'run', 'build:monorepo-ai-llm'],
    ['npm', 'run', 'build:users-api'],
    ['npm', 'run', 'build:llm-ops-api'],
    ['npm', 'run', 'build:sharepoint-api'],
    ['npm', 'run', 'build:sync-api'],
]

for cmd in commands:
    print('\n$', ' '.join(cmd))
    completed = subprocess.run(cmd, cwd=DESTINO, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f'Falha em: {cmd}')

print('\nTodas as builds executaram com sucesso.')



$ npm run build:monorepo-ai-llm

> monorepo-ai-llm@0.0.1 build:monorepo-ai-llm
> nest build monorepo-ai-llm

webpack 5.106.0 compiled successfully in 2234 ms


$ npm run build:users-api

> monorepo-ai-llm@0.0.1 build:users-api
> nest build users-api

webpack 5.106.0 compiled successfully in 2429 ms


$ npm run build:llm-ops-api

> monorepo-ai-llm@0.0.1 build:llm-ops-api
> nest build llm-ops-api

webpack 5.106.0 compiled successfully in 2895 ms


$ npm run build:sharepoint-api

> monorepo-ai-llm@0.0.1 build:sharepoint-api
> nest build sharepoint-api

webpack 5.106.0 compiled successfully in 2591 ms


$ npm run build:sync-api

> monorepo-ai-llm@0.0.1 build:sync-api
> nest build sync-api

webpack 5.106.0 compiled successfully in 3096 ms


Todas as builds executaram com sucesso.
